### Mugrade boilerplate

In [ ]:
### Run this cell to install and import the homework tests
!pip install --upgrade git+https://github.com/locuslab/mugrade.git
!wget -nc https://raw.githubusercontent.com/zkolter/llm_speedrun/refs/heads/main/part3_llm_training_tests.py

import mugrade
import os
from part3_llm_training_tests import *

def _mugrade_name(name):
    def rename(function):
        function.__name__ = name
        return function
    return rename

os.environ["MUGRADE_HW"] = "Part 3 - LLM Training"
os.environ["MUGRADE_KEY"] = "" ### Your key here

### BPE

Insert the BPE tokenizer you built in Part 1.

In [ ]:
### BEGIN YOUR CODE
pass
### END YOUR CODE

### LLM Architecture

Insert your LLM architecture from Part 2.

In [ ]:
### BEGIN YOUR CODE
pass
### END YOUR CODE

### Downloading necessary files

These next lines will donwload the necessary tokenizer and pre-tokenized data files, which you can build in Part 1 (but which require a fairly substantial machine to run as-is).

In [ ]:
from huggingface_hub import hf_hub_download
import os

repo = "zkolter/llm_speedrun"
filenames = ["fineweb-edu-10BT.shuffle.bin", "smoltalk.shuffle.bin", "tokenizer_50M.bpe"]

for filename in filenames:
    if not os.path.exists(filename):
        hf_hub_download(repo_id=repo, filename=filename, repo_type="dataset", local_dir=".")

You can use the following config file for training.  This is intended to load a batch size that will fit comfortably on a GPU with 80GB of memory, you can adjust as needed.

In [ ]:
%%writefile config.d12.json
{
    "depth": 12,
    "aspect_ratio": 64,
    "mlp_multiple": 4,
    "head_dim": 128,
    "dtype": "bfloat16",
    "vocab_size": 32768,
    "batch_size": 16,
    "seq_len": 2048,
    "rope_theta": 10000,
    "tokenizer": "tokenizer_50M.bpe",
    "token_multiple": 20,
    "lr": 7e-4,
    "weight_decay": 0.025,
    "data_mix": {
        "fineweb-edu-10BT.shuffle.bin": 0.875,
        "smoltalk.shuffle.bin": 0.125
    },
    "num_gpus": 1
}

### LLM Training

In [ ]:
import wandb
import time
from array import array
import os


# @mugrade.local_tests
def cross_entropy_loss(logits, y):
    ### BEGIN YOUR CODE
    logits = logits - logits.max(dim=-1, keepdim=True)[0]
    loss = -logits.take_along_dim(y.unsqueeze(-1), dim=-1)
    loss += logits.exp().sum(dim=-1, keepdim=True).log()
    return loss.mean()
    ### END YOUR CODE

class Adam:
    # @mugrade.local_tests
    @_mugrade_name("Adam_init")
    def __init__(self, params, lr_schedule, betas = (0.9, 0.95), eps=1e-5, weight_decay=0.0):
        ### BEGIN YOUR CODE
        self.schedule = lr_schedule
        self.betas = betas
        self.params = params
        self.t = 1
        self.weight_decay = weight_decay
        self.u = {k: torch.zeros_like(v) for k,v in params.items()}
        self.v = {k: torch.zeros_like(v) for k,v in params.items()}
        self.eps = eps
        for p in self.params.values():
            p.requires_grad_()
        ### END YOUR CODE

    # @mugrade.local_tests
    def step(self):
        ### BEGIN YOUR CODE
        with torch.no_grad():
          for k,p in self.params.items():
              self.u[k] = self.betas[0]*self.u[k] + (1-self.betas[0])*p.grad
              self.v[k] = self.betas[1]*self.v[k] + (1-self.betas[1])*p.grad**2
              p.grad.zero_()

          u_hat = self.u[k] / (1 - self.betas[0]**self.t)
          v_hat = self.v[k] / (1 - self.betas[1]**self.t)

          p *= (1 - self.schedule.get_lr(self.t) * self.weight_decay)
          p -= self.schedule.get_lr(self.t) * u_hat / (v_hat.sqrt() + self.eps)
        self.t += 1
        ### END YOUR CODE

class LRSchedule:
    # @mugrade.local_tests
    @_mugrade_name("LRSchedule_init")
    def __init__(self, total_steps, lr=1e-3, warmup_steps=50, decay_ratio=0.4, min_frac=0.1):
        ### BEGIN YOUR CODE
        self.total_steps = total_steps
        self.lr = lr
        self.warmup_steps = warmup_steps
        self.decay_steps = round(decay_ratio * total_steps)
        self.min_frac = min_frac
        ### END YOUR CODE

    # @mugrade.local_tests
    def get_lr(self, step):
        ### BEGIN YOUR CODE
        if step < self.warmup_steps:
            p = step / self.warmup_steps
            return (1-p)*self.min_frac*self.lr + p*self.lr
        elif step > self.total_steps - self.decay_steps:
            p = (step - (self.total_steps - self.decay_steps)) / self.decay_steps
            return (1-p)*self.lr + p*self.lr*self.min_frac
        else:
            return self.lr
        ### END YOUR CODE


# @mugrade.local_tests
def train_llm(config, log=False):
    ### BEGIN YOUR CODE
    llm = LLM(config)
    tokenizer = BPE(config["tokenizer"])
    for k in llm.params: llm.params[k] = llm.params[k].cuda()
    for k in llm.buffers: llm.buffers[k] = llm.buffers[k].cuda()

    config["total_params"] = sum(p.numel() for p in llm.params)
    config["total_tokens"] = config["total_params"] * config["token_multiple"]
    config["total_step"] = config["total_tokens"] / (config["batch_size"] * config["seq_len"])
    schedule = LRSchedule(config["total_step"], config["lr"])
    opt = Adam(llm.params, schedule, weight_decay=config["weight_decay"])
    if log:
        run = wandb.init(project="llm_speedrun", config=config)

    batch_items = []
    for k,p in config["data_mix"].items():
        batch_items += [(k, i*(config["seq_len"]+1)*2) for i in range(round(p*config["batch_size"]))]
    file_offsets = {k:0 for k in config["data_mix"]}
    n_tok = 0

    while opt.t < config["total_steps"]:
        start_time = time.perf_counter()
        tokens = []
        for filename, offset in batch_items:
            with open(filename, "rb") as f:
                f.seek(file_offsets[filename] + offset)
                tokens += array("H", f.read((config["seq_len"]+1)*2)).tolist()

        for k,p in config["data_mix"].items():
            read_size = round(p*config["batch_size"]) * (config["seq_len"]+1) * 2
            file_offsets[k] += read_size
            if file_offsets[k] + read_size >= os.path.getsize(k):
              file_offsets[k] = 0

    tokens = torch.tensor(tokens).reshape(config["batch_size"], config["seq_len"]+1)
    text = tokenizer.decode(tokens[:,1:].flatten()).tolist()

    # run and take a gradient
    tokens = tokens.cuda()
    logits = llm(tokens[:,:-1]).float()
    loss = cross_entropy_loss(logits, tokens[:,1:])
    loss.backward()
    opt.step()

    bpb = loss.item() * config["batch_size"] * config["seq_len"] / (len(text) * math.log(2))
    n_tok += config["batch_size"] * config["seq_len"]
    tok_per_sec = config["batch_size"] * config["seq_len"] / (time.perf_counter() - start_time)

    return llm

    ### END YOUR CODE

If your code passes all the tests, you can (optionally) uncomment this block to train the d12 model, here done on a single GPU.

In [ ]:
with open("config.d12.class.json", "rt") as f:
    config = json.load(f)
torch_types = {"float32": torch.float32, "bfloat16": torch.bfloat16}
config["dtype"] = torch_types[config["dtype"]]

llm = train_llm(config, log=True)

### Distributed LLM Training

Implement a distributed version of the training code, starting from the version above.  Note that there was one error in the class version, that `config["batch_size"]` should be replaced by `local_batch_size` in the line that computes bpb for logging.

In [ ]:
# @mugrade.local_tests
def train_llm_distributed(rank, nccl_uid, config, log=False):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE

If you're able to access a machine with 8 GPUs, then the following code will launch the distributed training run with a different config that uses a larger batch size.

In [ ]:
%%writefile config.d12.json
{
    "depth": 12,
    "aspect_ratio": 64,
    "mlp_multiple": 4,
    "head_dim": 128,
    "dtype": "bfloat16",
    "vocab_size": 32768,
    "batch_size": 128,
    "seq_len": 2048,
    "rope_theta": 10000,
    "tokenizer": "tokenizer_50M.bpe",
    "token_multiple": 20,
    "lr": 1e-3,
    "weight_decay": 0.1,
    "data_mix": {
        "fineweb-edu-10BT.shuffle.bin": 0.875,
        "smoltalk.shuffle.bin": 0.125
    },
    "num_gpus": 8
}

In [ ]:
from joblib import Parallel, delayed
os.environ["NCCL_NVLS_ENABLE"] = "0"  # shouldn't be needed if your system isn't messed up like mine

with open("config.d12.class.json", "rt") as f:
    config = json.load(f)
torch_types = {"float32": torch.float32, "bfloat16": torch.bfloat16}
config["dtype"] = torch_types[config["dtype"]]
nccl_uid = torch.cuda.nccl.unique_id()

Parallel(n_jobs = config["num_gpus"], backend="loky")(
    delayed(train_llm_distributed)(i, nccl_uid, config, log=True) for i in range(config["num_gpus"])
)
